# Notebook 03 — Explanatory Analysis
**Capstone Project CC26-PRU469**: Skill Gap Simulator

---

## Tujuan
Menjawab **3 Research Questions** dengan narasi dan visualisasi yang mendukung:

1. **RQ1**: Skill apa yang paling memengaruhi kesiapan kerja untuk role tertentu?
2. **RQ2**: Berapa estimasi waktu siap kerja berdasarkan profil user, jam belajar, dan skenario pasar?
3. **RQ3**: Bagaimana perubahan kondisi pasar memengaruhi prioritas pengembangan skill?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

PROCESSED = os.path.join('..', 'data', 'processed')
REPORTS = os.path.join('..', 'reports')

df_postings = pd.read_csv(os.path.join(PROCESSED, 'job_postings_cleaned.csv'))
df_job_skills = pd.read_csv(os.path.join(PROCESSED, 'job_skills_cleaned.csv'))
df_skill_master = pd.read_csv(os.path.join(PROCESSED, 'skill_master_cleaned.csv'))
df_users = pd.read_csv(os.path.join(PROCESSED, 'user_profiles_cleaned.csv'))
df_user_skills = pd.read_csv(os.path.join(PROCESSED, 'user_skills_cleaned.csv'))
df_freq = pd.read_csv(os.path.join(PROCESSED, 'skill_frequency.csv'))

roles = ['Data Analyst', 'Frontend Developer', 'UI/UX Designer']
colors = ['#2196F3', '#4CAF50', '#FF9800']
print('Data loaded.')

---

## RQ1: Skill apa yang paling memengaruhi kesiapan kerja?

Untuk menjawab pertanyaan ini, kita menganalisis dari **dua perspektif**:
1. **Supply side**: Skill mana yang paling sering diminta di job postings?
2. **Demand side**: Skill mana yang paling sering menjadi *missing skill* bagi user?

Skill yang **sering diminta** DAN **sering menjadi gap** adalah skill paling krusial.

In [ ]:
# Analisis 1: Top skills dari job postings (importance score)
fig, axes = plt.subplots(1, 3, figsize=(22, 8))

for i, role in enumerate(roles):
    sm = df_skill_master[df_skill_master['role'] == role].sort_values('importance_score', ascending=True)
    palette = 'Blues' if i == 0 else 'Greens' if i == 1 else 'Oranges'
    
    bars = axes[i].barh(sm['skill_name'], sm['importance_score'],
                        color=sns.color_palette(palette, len(sm)))
    axes[i].set_title(f'{role}\nImportance Score', fontweight='bold', fontsize=12)
    axes[i].set_xlabel('Importance Score (0-1)')
    axes[i].set_xlim(0, 1.1)
    
    # Annotate values
    for bar, val in zip(bars, sm['importance_score']):
        axes[i].text(val + 0.02, bar.get_y() + bar.get_height()/2, 
                     f'{val:.2f}', va='center', fontsize=8)

plt.suptitle('RQ1: Importance Score per Skill — Seberapa Penting Skill Ini di Pasar?',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'expl_01_skill_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Analisis 2: Cross-reference dengan missing skills
# Hitung frekuensi setiap skill muncul sebagai top missing skill
all_missing = []
for col in ['top_missing_skill_1', 'top_missing_skill_2', 'top_missing_skill_3']:
    temp = df_users[['target_role', col]].rename(columns={col: 'skill'})
    all_missing.append(temp)

df_missing = pd.concat(all_missing)
missing_freq = df_missing.groupby(['target_role', 'skill']).size().reset_index(name='missing_count')

fig, axes = plt.subplots(1, 3, figsize=(22, 8))

for i, role in enumerate(roles):
    subset = missing_freq[missing_freq['target_role'] == role].nlargest(10, 'missing_count')
    subset = subset.sort_values('missing_count', ascending=True)
    palette = 'Reds' if True else 'Blues'
    
    bars = axes[i].barh(subset['skill'], subset['missing_count'],
                        color=sns.color_palette('Reds_r', len(subset)))
    axes[i].set_title(f'{role}\nFrekuensi Sebagai Missing Skill', fontweight='bold')
    axes[i].set_xlabel('Frekuensi Muncul Sebagai Gap')

plt.suptitle('RQ1: Skill Mana yang Paling Sering Menjadi Gap bagi User?',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'expl_02_missing_skill_freq.png'), dpi=150, bbox_inches='tight')
plt.show()

### Insight RQ1

**Temuan Utama:**
- Untuk **Data Analyst**: SQL dan Python memiliki importance tertinggi, dan juga sering menjadi missing skill. Ini menunjukkan bahwa kedua skill ini adalah *critical path* menuju kesiapan kerja.
- Untuk **Frontend Developer**: JavaScript dan React adalah skill paling krusial — baik dari sisi demand maupun gap.
- Untuk **UI/UX Designer**: Figma dan User Research adalah skill yang paling memengaruhi kesiapan.

**Implikasi**: Prioritas belajar harus difokuskan pada skill dengan importance tinggi yang juga merupakan gap terbesar.

---

## RQ2: Berapa estimasi waktu siap kerja?

Estimasi waktu dipengaruhi oleh **3 faktor utama**:
1. Gap score (seberapa besar gap skill)
2. Jam belajar per minggu
3. Skenario pasar (Normal, Kompetitif, Booming)

In [ ]:
# Analisis 3: Estimasi waktu berdasarkan background level
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

order = ['Pemula', 'Menengah', 'Lanjutan']

# Rata-rata estimasi per background x role
pivot_weeks = df_users.pivot_table(values='estimated_weeks_ready', 
                                    index='background_level',
                                    columns='target_role', 
                                    aggfunc='mean')
pivot_weeks = pivot_weeks.reindex(order)

pivot_weeks.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Rata-rata Estimasi Minggu per Background Level', fontweight='bold')
axes[0].set_ylabel('Minggu')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(fontsize=9)

# Per scenario
pivot_scenario = df_users.pivot_table(values='estimated_weeks_ready',
                                      index='market_scenario',
                                      columns='target_role',
                                      aggfunc='mean')
pivot_scenario.plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('Rata-rata Estimasi Minggu per Skenario Pasar', fontweight='bold')
axes[1].set_ylabel('Minggu')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(fontsize=9)

plt.suptitle('RQ2: Estimasi Waktu Siap Kerja', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'expl_03_time_estimation.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Analisis 4: Pengaruh study hours terhadap estimated weeks
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, role in enumerate(roles):
    subset = df_users[df_users['target_role'] == role]
    scatter = axes[i].scatter(subset['study_hours_per_week'], 
                              subset['estimated_weeks_ready'],
                              c=subset['gap_score'], cmap='RdYlGn_r',
                              alpha=0.6, s=30, edgecolors='gray', linewidths=0.3)
    axes[i].set_title(f'{role}', fontweight='bold')
    axes[i].set_xlabel('Study Hours/Week')
    axes[i].set_ylabel('Estimated Weeks' if i == 0 else '')
    plt.colorbar(scatter, ax=axes[i], label='Gap Score')

plt.suptitle('RQ2: Study Hours vs Estimated Weeks (warna = Gap Score)',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'expl_04_study_hours_effect.png'), dpi=150, bbox_inches='tight')
plt.show()

### Insight RQ2

**Temuan Utama:**
- **Pemula** membutuhkan waktu paling lama (rata-rata ~100+ minggu), **Lanjutan** hanya ~30-50 minggu.
- **Skenario Kompetitif** menambah ~30% waktu persiapan dibanding Normal.
- **Skenario Booming** mengurangi ~20% waktu (demand tinggi, standar lebih fleksibel).
- **Study hours lebih tinggi** secara signifikan mempersingkat estimasi waktu.

---

## RQ3: Bagaimana perubahan kondisi pasar memengaruhi prioritas skill?

Setiap skenario pasar mengubah bobot importance skill secara berbeda.

In [ ]:
# Analisis 5: Perubahan ranking skill antar skenario
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

for i, role in enumerate(roles):
    sm = df_skill_master[df_skill_master['role'] == role].copy()
    
    # Hitung selisih competitive vs booming
    sm['delta'] = sm['market_demand_competitive'] - sm['market_demand_booming']
    sm = sm.sort_values('delta', ascending=True)
    
    bar_colors = ['#FF5722' if d > 0 else '#4CAF50' for d in sm['delta']]
    axes[i].barh(sm['skill_name'], sm['delta'], color=bar_colors)
    axes[i].axvline(x=0, color='black', linewidth=0.8)
    axes[i].set_title(f'{role}', fontweight='bold')
    axes[i].set_xlabel('Selisih (Kompetitif - Booming)')

plt.suptitle('RQ3: Pergeseran Prioritas Skill Antar Skenario\n(Merah = lebih penting di Kompetitif, Hijau = lebih penting di Booming)',
             fontweight='bold', fontsize=13, y=1.05)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'expl_05_scenario_shift.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Analisis 6: Gap score distribution per skenario pasar
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scenarios = ['Normal', 'Kompetitif', 'Booming']
scenario_colors = ['#2196F3', '#FF5722', '#4CAF50']

for i, role in enumerate(roles):
    subset = df_users[df_users['target_role'] == role]
    for sc, sc_color in zip(scenarios, scenario_colors):
        sc_data = subset[subset['market_scenario'] == sc]
        axes[i].hist(sc_data['gap_score'], bins=20, alpha=0.5, label=sc, color=sc_color)
    axes[i].set_title(f'{role}', fontweight='bold')
    axes[i].set_xlabel('Gap Score')
    axes[i].set_ylabel('Jumlah User' if i == 0 else '')
    axes[i].legend(fontsize=9)

plt.suptitle('RQ3: Distribusi Gap Score per Skenario Pasar', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'expl_06_gap_by_scenario.png'), dpi=150, bbox_inches='tight')
plt.show()

### Insight RQ3

**Temuan Utama:**
- Dalam **skenario Kompetitif**, semua skill mendapat bobot lebih tinggi — user harus lebih lengkap.
- Dalam **skenario Booming**, bobot skill lebih rendah — perusahaan lebih fleksibel, fokus pada core skills saja.
- Skill dengan importance awalnya tinggi (SQL, JavaScript, Figma) mengalami pergeseran paling besar antar skenario.

**Implikasi**: Strategi belajar harus adaptif terhadap kondisi pasar — di pasar kompetitif, breadth penting; di pasar booming, depth lebih efektif.

---

## Kesimpulan Explanatory Analysis

| Research Question | Jawaban Singkat |
|---|---|
| RQ1: Skill paling berpengaruh? | SQL & Python (DA), JavaScript & React (FE), Figma & User Research (UI/UX) |
| RQ2: Estimasi waktu siap kerja? | Pemula: ~100 minggu, Menengah: ~60 minggu, Lanjutan: ~30 minggu (skenario Normal) |
| RQ3: Pengaruh kondisi pasar? | Kompetitif menambah ~30% waktu, Booming mengurangi ~20% waktu |